# SINDy - Two Variable

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp

### Library Construction

In [ ]:
def createLibrary(x, y,polynomial_degree):
  # columns of PHI
  n = polynomial_degree
  n_cols = 1 + 2 * n + (n-1)*(n)//2
  PHI = np.zeros((len(x), n_cols))

  #1
  PHI[:,0] = 1
  # x^j, assume x is a column vector
  powers = np.arange(1,polynomial_degree+1)
  X_block = x**powers
  PHI[:,1:polynomial_degree+1] = X_block

  # y^j, assumes y is a column vector (m x 1)
  Y_block = y**powers
  PHI[:,polynomial_degree+1:2*polynomial_degree+1] = Y_block

  # x^k.y^j
  ## total degree from 2, 3,.... , polynomial_degree
  start_column = 2*polynomial_degree+1
  for t in range(2,polynomial_degree+1):
    ## total degree = t
    ### X_block = [x^(t-1) X^(t-2) .... X^2      X^1    ]
    ### Y_block = [Y^1     Y^2     .... Y^(t-2)  Y^(t-1)]

    powers_x = np.arange(t-1,0,-1) # t-1 t-2 .... 2   1
    powers_y = np.arange(1,t,1)    # 1    2       t-2 t

    X_block = x**powers_x
    Y_block = y**powers_y

    # multiplying X, Y blocks to get XY = [x^j.y^k]  where j + k = t, (j = 1 to t-2)
    XY_t = X_block* Y_block

    #
    PHI[:, start_column:start_column+XY_t.shape[1]] = XY_t
    start_column += XY_t.shape[1]

  return PHI


## Generating Data from Models

### Simple Linear Model

$$\dot{x} = -2x+y $$
$$\dot{y} = x - 3y$$

In [1]:
def generate_linear():
    def model(t, state):
        x, y = state
        dxdt = -x + 2*y
        dydt = -3*x - y
        return [dxdt, dydt]

    t = np.linspace(0, 10, 1000)

    solution = solve_ivp(model, [t[0], t[-1]], [1, 1], t_eval=t)

    x = solution.y[0]
    y = solution.y[1]

    dxdt = -x + 2*y
    dydt = -3*x - y

    return x, y, dxdt, dydt

### Lotka Volterra

In [2]:
def generate_lotka_volterra():
    def model(t, state):
        x, y = state
        dxdt = x - x*y
        dydt = x*y - y
        return [dxdt, dydt]

    t = np.linspace(0, 20, 2000)

    solution = solve_ivp(model, [t[0], t[-1]], [2, 1], t_eval=t)

    x = solution.y[0]
    y = solution.y[1]

    dxdt = x - x*y
    dydt = x*y - y

    return x, y, dxdt, dydt

### Van Der Pol Oscillator

In [3]:
def generate_van_der_pol():
    def model(t, state):
        x, y = state
        dxdt = y
        dydt = y - x - x**2 * y
        return [dxdt, dydt]

    t = np.linspace(0, 20, 2000)

    solution = solve_ivp(model, [t[0], t[-1]], [2, 0], t_eval=t)

    x = solution.y[0]
    y = solution.y[1]

    dxdt = y
    dydt = y - x - x**2 * y

    return x, y, dxdt, dydt

### Duffing Oscillator

In [4]:
def generate_duffing():
    def model(t, state):
        x, y = state
        dxdt = y
        dydt = x - 0.2*y - x**3
        return [dxdt, dydt]

    t = np.linspace(0, 30, 3000)

    solution = solve_ivp(model, [t[0], t[-1]], [1, 0], t_eval=t)

    x = solution.y[0]
    y = solution.y[1]

    dxdt = y
    dydt = x - 0.2*y - x**3

    return x, y, dxdt, dydt

### Brusselator

In [5]:
def generate_brusselator():
    def model(t, state):
        x, y = state
        dxdt = 1 - 4*x + x**2 * y
        dydt = 3*x - x**2 * y
        return [dxdt, dydt]

    t = np.linspace(0, 30, 3000)

    solution = solve_ivp(model, [t[0], t[-1]], [1.5, 3], t_eval=t)

    x = solution.y[0]
    y = solution.y[1]

    dxdt = 1 - 4*x + x**2 * y
    dydt = 3*x - x**2 * y

    return x, y, dxdt, dydt

## Sparse Regression Function

In [ ]:
def SparseRegression(PHI, x_dot, n_iterations, lamb):
  ## Ordinary Least Squares
  # E - coefficient vector
  E,*_ = np.linalg.lstsq(PHI, x_dot, rcond = None)
  #print(E)
  ### iterations
  for i in range(n_iterations):
    ## getting the idices |e_j| < lambda
    smallIndices = np.abs(E)<lamb
    ## setting small coefficients to 0
    E[smallIndices] = 0
    ## doing linear regression with the remaining coefficients
    E[~smallIndices],*_ = np.linalg.lstsq(PHI[:,~smallIndices], x_dot, rcond = None)
  return E

### Testing Sparse Regression